In [1]:
import spacy

# Load the pre-trained English NER model
nlp = spacy.load("en_core_web_md")


In [2]:
import csv
from pathlib import Path
import json

def read_csv(csv_path):
    
    with csv_path.open(encoding = "utf-8") as f:
        reader = csv.DictReader(f)
        articles = []
        
        for row in reader:
            pid, sections = row
            print(row[pid])
            sections = json.loads(row["sections"])
            pid = row[pid]
            articles.append({"pid": pid, "sections": sections})

    return articles

articles = read_csv(Path("articles_sections_acknowledgments.csv"))

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27

In [3]:
import re

def clean_scientific_text(text):

    # divide by sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    # remove short sentences
    sentences = [sent.strip() for sent in sentences if len(sent.strip()) > 20]

    cleaned_sentences = []
    for sent in sentences:
        # remove sentences with lots of math symbols (heuristic)
        if re.search(r'[=<>+\*/^|]', sent):
            #sent = "########### deleted ##############"
            continue
        if re.search(r'\d+\s*=\s*\d+', sent):
            #sent = "########### deleted ##############"
            continue
        if len(sent.strip()) < 10:
            #sent = "########### deleted ##############"
            continue
            
        doc = nlp(sent)
        if not any(token.pos_ == "VERB" for token in doc):
            continue
        cleaned_sentences.append(sent.strip())

    cleaned_text = " ".join(cleaned_sentences)
    return cleaned_text

for article in articles:
    sect_names = list(article.get("sections").keys())
    for sn in sect_names:
        texttt = clean_scientific_text(article.get("sections").get(sn))
        article["sections"][sn] = texttt

    print(f"Processing article {article.get("pid")}")




Processing article 2
Processing article 3
Processing article 4
Processing article 5
Processing article 6
Processing article 7
Processing article 8
Processing article 9
Processing article 10
Processing article 11
Processing article 12
Processing article 13
Processing article 14
Processing article 15
Processing article 16
Processing article 17
Processing article 18
Processing article 19
Processing article 20
Processing article 21
Processing article 22
Processing article 23
Processing article 24
Processing article 25
Processing article 26
Processing article 27
Processing article 28
Processing article 29
Processing article 30
Processing article 31
Processing article 32
Processing article 33
Processing article 34
Processing article 35
Processing article 36
Processing article 37
Processing article 38
Processing article 39
Processing article 40
Processing article 41
Processing article 42
Processing article 43
Processing article 44
Processing article 45
Processing article 46
Processing article

In [8]:
import os
import re
from rdflib import Graph, Namespace, Literal, RDF, URIRef, OWL

RPO  = Namespace("http://www.semanticweb.org/ftsdemo/ontologies/2025/5/rpo#")
mentions = RPO.mentions
observed_in_sentence = RPO.observed_in_sentence
has_content = RPO.has_content
observed_in_section = RPO.observed_in_section

g = Graph()
g.bind("rpo", RPO)
all_ents = 0
not_skipped = []
entities = []
per = 0
org = 0
geo = 0
misc = 0
short = 0

short_ents = {}

def fix_ligatures(text):
    ligatures = {
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬃ": "ffi",
        "ﬄ": "ffl",
        "ﬀ": "ff",
        "ﬆ": "st"
    }
    for lig, replacement in ligatures.items():
        text = text.replace(lig, replacement)
    return text

def makeName(string):
    org = False
    string = string.replace('&', 'and')
    words = re.findall(r'\b[A-Za-z0-9]+\b', string)
    result = []
    for word in words:
        if word.isupper():
            result.append(word)
        else:
            result.append(word.title())

    return ''.join(result)

#process each text file
for article in articles:
    print(f"======================================== analysing paper {article.get('pid')} ======================================================= \n")

    paper_uri = RPO[f"paper/{article.get('pid')}"]
    section_titles = list(article.get("sections").keys())
    if "References" in section_titles:
        section_titles.remove("References")
    if "Bibliography" in section_titles:
        section_titles.remove("Bibliography")
        

    for sect in list(section_titles):
        doc = nlp(article.get("sections").get(sect))
        print(" #################################", sect, " ################################# ")
        sect_uri = RPO[makeName(sect)]

        for sent in doc.sents:
            sentence_uri = URIRef(paper_uri + "/sent" + str(sent.start_char))
            for ent in sent.ents:
                if ent.label_ in ["ORG", "GPE", "LOC", "FAC", "NORP", "PRODUCT", "EVENT", "WORK_OF_ART"]:
                    all_ents += 1
                    ent_text_raw = fix_ligatures(ent.text.strip())
                    #print("ent text raw", ent_text_raw)

                    # Skip if mostly symbols or doesn't resemble a word
                    num_letters = len(re.findall(r'[A-Za-z]', ent_text_raw))
                    num_symbols = len(re.findall(r'[^A-Za-z0-9]', ent_text_raw))

                    if num_letters < 2 or num_symbols >= num_letters:
                        continue  # Skip entities made mostly of symbols

                    if ent_text_raw.lower().startswith('the '):
                        ent_text_raw = ent_text_raw[4:] 
                    ent_cl = re.sub(r'[^A-Za-z0-9 ]+', ' ', ent_text_raw).strip().title()
                    ent_clean = re.sub(r'[^A-Za-z0-9 ]+', '', ent_text_raw).strip()
                    #print("ent clean", ent_clean)
                    words = ent_clean.split(" ")
                    #print("words", words)

                    if ent_clean.isupper() and " " not in ent_clean:
                        ent_final = ent_clean  # Acronym (e.g., NASA)
                        ent_name = ent_clean
                    elif len(words) > 1:
                        ent_final = '_'.join(word.capitalize() for word in words)
                        ent_name = ''.join(word.capitalize() for word in words)
                    else:
                        # Keep single word as-is (after stripping)
                        ent_final = ent_clean.capitalize()
                        ent_name = ent_clean.capitalize()
                    if len(ent_final) < 3 and ent_final not in ["UN", "EU", "UK", "US"]:
                        print(ent_final, "not added, too short")
                        continue
                    
                    if ent_final not in entities:
                        entities.append(ent_final)
                        ent_uri = RPO[ent_name]
                            
                        # RDF typing

                        #person skipped, too problematic
                        #if ent.label_ == "PERSON":
                        #    g.add((ent_uri, RDF.type, RPO.Person))
                        #    print(f"added {ent_name} as Person")
                        #    per += 1
                        if ent.label_ in ["ORG", "NORP"]:
                            if "cknowledgment" in sect:
                                g.add((ent_uri, RDF.type, RPO.Fund))   
                                g.add((paper_uri, RPO.funded_by, ent_uri))
                            else:
                                g.add((ent_uri, RDF.type, RPO.Organisation))
                            print(f"added {ent_name} as Organisation ({ent.label_})")
                            #print(ent_name, ent_uri)
                            org += 1
                        elif ent.label_ in ["GPE", "LOC"]:
                            if "cknowledgment" in sect:
                                g.add((ent_uri, RDF.type, RPO.Fund))
                                g.add((paper_uri, RPO.funded_by, ent_uri))
                            else:
                                g.add((ent_uri, RDF.type, RPO.Geographical))
                            print(f"added {ent_name} as Geographical")
                            #print(ent_name, ent_uri)
                            geo += 1
                        elif ent.label_ in ["PRODUCT", "FAC", "EVENT", "WORK_OF_ART"]:
                            if "cknowledgment" in sect:
                                g.add((ent_uri, RDF.type, RPO.Fund))
                                g.add((paper_uri, RPO.funded_by, ent_uri))
                            else:
                                g.add((ent_uri, RDF.type, RPO.Thing))
                            print(f"added {ent_name} as Thing ({ent.label_})")
                            #print(ent_name, ent_uri)
                            misc += 1

                        not_skipped.append(ent_final)
                        #add graph triples
                        if "cknowledgment" not in sect:
                            g.add((paper_uri, mentions, ent_uri))
                            g.add((paper_uri, RDF.type, RPO.AcademicWork))
                            g.add((ent_uri, observed_in_sentence, sentence_uri))
                            g.add((sentence_uri, has_content, Literal(sent.text)))
                            g.add((sentence_uri, RDF.type, RPO.Sentence))
                            g.add((ent_uri, observed_in_section, sect_uri))
                            g.add((sentence_uri, observed_in_section, sect_uri))
                            g.add((sect_uri, RDF.type, RPO.Section))
                        else:
                            print(f"Acknowledgment, added {ent_name} only as funder")
                    else:
                        if "cknowledgment" not in sect:
                            ent_uri = RPO[makeName(ent_name)]
                            g.add((ent_uri, observed_in_sentence, sentence_uri))
                            g.add((sentence_uri, has_content, Literal(sent.text)))
                            g.add((sentence_uri, RDF.type, RPO.Sentence))
                            g.add((ent_uri, observed_in_section, sect_uri))
                            g.add((sentence_uri, observed_in_section, sect_uri))
                            g.add((sect_uri, RDF.type, RPO["Section"]))
                        else:
                            print(f"Acknowledgment, added {ent_name} only as funder")

                        #print info
                    #print(f"added{ent_final}, {ent.label}, {sent.text}")


======================================== analysing paper 2 ======================================================= 

 ################################# Abstract  ################################# 
 ################################# 1 Motivation  ################################# 
added Field as Organisation (ORG)
added OntologyAlignmentEvaluationInitiative as Organisation (ORG)
 ################################# 2 Requirements for User Support in Large-Scale Ontology Alignment  ################################# 
added Ter as Organisation (ORG)
added RepresentationAndInteraction as Organisation (ORG)
added SAMBO as Organisation (ORG)
added GOMMA as Organisation (ORG)
added Largebio as Thing (PRODUCT)
 ################################# 3 Literature Study  ################################# 
 ################################# 4 User Interface Evaluations  ################################# 
added SUS as Organisation (ORG)
added CMinimizeTheUsersMemoryLoad as Organisation (ORG)
added IPreven

In [9]:
out_file = "NER_dirty.ttl"
g.serialize(destination=out_file, format="turtle")

<Graph identifier=Nf2ef831f541242718bf00981d92f7d00 (<class 'rdflib.graph.Graph'>)>

In [10]:
print(org)
print(geo)
print(misc)

21251
2662
4604
